# Let's build GPT（导读）

整个系列的正题。从一个查表 bigram 出发，一路加到一个真正的 decoder-only Transformer。

**这一章换语料**：tiny Shakespeare，1,115,394 个字符，词表 65（含换行、标点、大小写）。
按位置切前 90% 训练、后 10% 验证——文本是连续的，**不能打乱**，
打乱会把同一句话的前后半分到两边，val loss 虚低。

`nnzh/shakespeare.py` 里已经备好 `load_text / build_vocab / train_val_split / get_batch` 四个函数，
逻辑一共二十来行，建议先扫一眼再往下走。

**基线**：uniform = log2(65) = **6.022 bpc**。这是什么都不学时的下限。

**硬件**：本机 RTX 4060 / 8.6GB。前六节在 CPU 上就能跑，第 6 节 scale up 必须上 GPU。


## 0 — Boilerplate

数据、词表、切分、`get_batch`。跑一下确认 `device` 是 `cuda`。

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from nnzh.shakespeare import (load_text, build_vocab, train_val_split,
                              encode, decode, get_batch)

text = load_text()
stoi, itos = build_vocab(text)
vocab_size = len(stoi)
train_data, val_data = train_val_split(text, stoi)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
print(f'{len(text):,} chars, vocab {vocab_size}, train {len(train_data):,}, val {len(val_data):,}, device {device}')


## 1 — Bigram 基线

先把 03 章那个 bigram 用 `nn.Module` 重写一遍，作为要打败的靶子。

模型就是一张 `nn.Embedding(vocab_size, vocab_size)` 查找表：
输入字符的 id 直接查出下一个字符的 logits，**完全不看更早的历史**。

三个这一章会反复用到的约定，在这里先立好：

- `forward(idx, targets=None)` 同时返回 `logits, loss`，`targets=None` 时只前向
- 算 loss 前要把 `(B, T, C)` 摊平成 `(B*T, C)`——`F.cross_entropy` 要求类别在第 1 维
- `generate(idx, max_new_tokens)` 每次取最后一个位置的 logits，`multinomial` 采样，拼回去

训练前的 loss 应该约等于 `ln 65 = 4.174`（nats）。对不上说明初始化有问题。


## 2 — 数学技巧：从"平均过去"到矩阵乘

这一节没有神经网络，只是一个恒等式，但它是整章的地基。

想让位置 `t` 看到 `0..t` 的历史，最笨的办法是求平均。三种写法**完全等价**，
从慢到快：

```
写法一  双重 for 循环，x[b,t] = mean(x[b,:t+1])
写法二  下三角矩阵 a（每行归一化）做 a @ x
写法三  wei = zeros(T,T); wei.masked_fill(tril==0, -inf); wei = softmax(wei, -1); wei @ x
```

**为什么绕到写法三**：前两种里的权重是固定的常数（均匀平均），
而写法三里 `wei` 在 softmax 之前是可以被**算出来**的——
一旦它由数据决定，均匀平均就变成了 self-attention。

`masked_fill(tril == 0, float('-inf'))` 是 decoder 的关键：
未来的位置填 `-inf`，softmax 之后权重恰好是 0。**这一步必须在 softmax 之前**，
之后再 mask 就归一化错了。


## 3 — 单头 self-attention

现在让权重由数据算出来。每个位置发出三个向量：

```
query  我在找什么
key    我有什么
value  如果你选中我，我给你什么
```

权重 = query 和 key 的点积，`wei = q @ k.transpose(-2,-1)`，再 mask、softmax，
最后 `out = wei @ v`。

**缩放因子 `head_size ** -0.5` 不是可选项。** q、k 是单位方差时，
点积的方差是 `head_size`；不除的话 `wei` 数量级会随 head_size 变大，
softmax 被推向饱和，退化成"只看一个位置"的 one-hot，梯度也跟着消失。
自己试一下去掉它，看 `wei` 的分布怎么塌掉。

几个容易记混的点：

- **注意力里没有位置概念**。它是在一个集合上做的，谁在前谁在后它不知道，
  所以要单独加 position embedding（`nn.Embedding(block_size, n_embd)`）
- batch 里的每个样本各算各的，样本之间永远不通信
- decoder 才需要 tril；做情感分类那种任务不 mask，就是 encoder
- self-attention 是 q/k/v 来自同一个 x；cross-attention 的 k/v 来自别处


## 4 — 多头 + FeedForward

**多头**：把 `n_embd` 切成若干份并行做注意力，再 concat 起来。
4 个 head、每个 head_size=8，和 1 个 head_size=32 参数量相近，
但前者能同时关注多种不同的关系。

**FeedForward**：`Linear -> ReLU -> Linear`，逐位置独立。

为什么需要它：注意力只是**收集信息**（做加权平均），收完之后还没有做过任何非线性的
"思考"。连着堆注意力等于连着做线性组合。中间那层的宽度是 `4 * n_embd`，
来自原论文。


## 5 — Block：残差 + LayerNorm

把「多头注意力 + FeedForward」打包成 Block，然后堆起来。直接堆会训不动，
需要两个东西：

**残差连接** `x = x + self.sa(x)`。梯度沿加法自由地流回输入，
深网络才能优化。注意投影层（每个子层输出后的那个 `Linear`）也是这里加的。

**LayerNorm**，而且是 **pre-norm**：

```python
x = x + self.sa(self.ln1(x))     # Karpathy / 现代做法：norm 在子层之前
x = x + self.ffwd(self.ln2(x))
```

原论文是 post-norm（`x = ln(x + sublayer(x))`）。**pre-norm 更好训**，
现在基本是标准做法。这是视频里少数几处明确偏离原论文的地方。

LayerNorm 和你 04 章写的 BatchNorm 长得几乎一样，区别只在**统计哪一维**：
BatchNorm 跨样本（列），LayerNorm 跨特征（行）。所以 LayerNorm 不需要
running buffer、不需要 train/eval 开关、batch size 为 1 也能用——
04 章那些坑在这里全部消失。


## 6 — Dropout + scale up

加 Dropout（注意力权重之后、每个子层输出之后），然后把规模拉到视频的配置：

```
batch_size = 64     block_size = 256
n_embd = 384        n_head = 6      n_layer = 6
dropout = 0.2       lr = 3e-4       max_iters = 5000
```

约 10M 参数。视频里 A100 跑 15 分钟，**4060 上估计要久得多**，
先用小配置把代码跑通，确认 loss 在降，再开这一档。

显存不够就先降 `batch_size`，它对显存的影响最直接。


## 7 — 生成 & 记录

采样 500 个 token 看效果。视频最终 val loss 约 1.48 nats。

换算成 bpc 记进 `experiments/bpc.md`——**单开一张表**，
和前五章的 names.txt 不同语料，不能放同一张表里比。


## 附：容易踩的地方

- `block_size` 是**最大**上下文长度。`generate` 的时候 idx 会越来越长，
  必须裁成 `idx[:, -block_size:]` 再喂进去，否则 position embedding 越界
- position embedding 用的是 `torch.arange(T, device=device)`——
  忘了 `device=` 会在 GPU 上报 device mismatch
- `wei` 的 mask 要写成 `self.tril[:T, :T]`，因为最后一个 batch 的 T 可能小于 block_size
- `tril` 用 `register_buffer` 注册，不是参数，但要跟着 `.to(device)` 走
- 估 loss 用 `@torch.no_grad()` + `model.eval()`，多个 batch 取平均——
  单个 batch 的 loss 噪声大到看不出趋势
